# Census Income ML Modeling - Databricks

This notebook implements the Machine Learning layer for the curated Delta table `workshop.default.donation_data_v1`.

Covered sections:

- `4.1` Data Preparation
- `4.2` Modeling
- `4.3` Manual Tuning
- `4.4` Evaluation
- `4.5` Experiment Tracking with MLflow

## 4.1 Data Preparation

- Load curated dataset
- Identify categorical and numerical features
- Handle missing values
- Encode categorical variables
- Scale numerical features

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Imputer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import mlflow
import mlflow.spark
from datetime import datetime

dbutils.widgets.text("source_table", "workshop.default.donation_data_v1")
dbutils.widgets.text("experiment_name", "/Shared/census_income_prediction")

source_table = dbutils.widgets.get("source_table")
experiment_name = dbutils.widgets.get("experiment_name")

mlflow.set_experiment(experiment_name)

print("Source table:", source_table)
print("MLflow experiment:", experiment_name)


In [ ]:
curated_df = spark.table(source_table)

display(curated_df.limit(20))
curated_df.printSchema()
print("Rows:", curated_df.count())


In [ ]:
label_col = "income"

numeric_features = [
    "age",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

categorical_features = [
    "workclass",
    "education_level",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country",
    "random_flag",
    "source_system"
]

model_columns = [label_col] + numeric_features + categorical_features

model_df = curated_df.select(*model_columns)
model_df = model_df.filter(F.col(label_col).isin("<=80k", ">80k"))

for column_name in categorical_features:
    model_df = model_df.withColumn(column_name, F.coalesce(F.col(column_name), F.lit("Unknown")))

print("Model rows after valid-label filter:", model_df.count())
display(model_df.groupBy(label_col).count())


In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())


In [ ]:
label_indexer = StringIndexer(
    inputCol=label_col,
    outputCol="label",
    handleInvalid="skip",
    stringOrderType="alphabetAsc"
)

numeric_imputer = Imputer(
    inputCols=numeric_features,
    outputCols=[f"{c}_imputed" for c in numeric_features],
    strategy="median"
)

categorical_indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_features
]

one_hot_encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_features],
    outputCols=[f"{c}_ohe" for c in categorical_features],
    handleInvalid="keep"
)

numeric_assembler = VectorAssembler(
    inputCols=[f"{c}_imputed" for c in numeric_features],
    outputCol="numeric_features"
)

numeric_scaler = StandardScaler(
    inputCol="numeric_features",
    outputCol="scaled_numeric_features",
    withMean=True,
    withStd=True
)

feature_assembler = VectorAssembler(
    inputCols=["scaled_numeric_features"] + [f"{c}_ohe" for c in categorical_features],
    outputCol="features"
)

preprocessing_stages = [
    label_indexer,
    numeric_imputer,
] + categorical_indexers + [
    one_hot_encoder,
    numeric_assembler,
    numeric_scaler,
    feature_assembler,
]


## 4.2 Modeling

- Train baseline models
- Use Logistic Regression, Decision Tree, and XGBoost when available
- Observe underfitting and overfitting by comparing train and test metrics

In [ ]:
try:
    from xgboost.spark import SparkXGBClassifier
    xgboost_available = True
    print("Spark XGBoost is available in this cluster.")
except Exception as exc:
    SparkXGBClassifier = None
    xgboost_available = False
    print("Spark XGBoost is not available. The notebook will use GBTClassifier as a tree-boosting fallback.")
    print("Reason:", str(exc))


## 4.3 Manual Tuning

The list below manually changes hyperparameters. Each configuration becomes one MLflow run, so we can compare performance across runs.

In [ ]:
model_configs = [
    {
        "run_name": "lr_baseline",
        "model_type": "LogisticRegression",
        "params": {"maxIter": 30, "regParam": 0.0, "elasticNetParam": 0.0},
        "estimator": LogisticRegression(featuresCol="features", labelCol="label", maxIter=30, regParam=0.0, elasticNetParam=0.0)
    },
    {
        "run_name": "lr_regularized",
        "model_type": "LogisticRegression",
        "params": {"maxIter": 60, "regParam": 0.05, "elasticNetParam": 0.2},
        "estimator": LogisticRegression(featuresCol="features", labelCol="label", maxIter=60, regParam=0.05, elasticNetParam=0.2)
    },
    {
        "run_name": "decision_tree_depth_5",
        "model_type": "DecisionTreeClassifier",
        "params": {"maxDepth": 5, "minInstancesPerNode": 10},
        "estimator": DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5, minInstancesPerNode=10, seed=42)
    },
    {
        "run_name": "decision_tree_depth_10",
        "model_type": "DecisionTreeClassifier",
        "params": {"maxDepth": 10, "minInstancesPerNode": 5},
        "estimator": DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=10, minInstancesPerNode=5, seed=42)
    }
]

if xgboost_available:
    model_configs.append(
        {
            "run_name": "xgboost_manual_tuned",
            "model_type": "SparkXGBClassifier",
            "params": {"max_depth": 5, "n_estimators": 80, "learning_rate": 0.1, "num_workers": 1},
            "estimator": SparkXGBClassifier(
                features_col="features",
                label_col="label",
                prediction_col="prediction",
                probability_col="probability",
                raw_prediction_col="rawPrediction",
                max_depth=5,
                n_estimators=80,
                learning_rate=0.1,
                num_workers=1,
                seed=42
            )
        }
    )
else:
    model_configs.append(
        {
            "run_name": "gbt_fallback_for_xgboost",
            "model_type": "GBTClassifier",
            "params": {"maxDepth": 5, "maxIter": 50, "stepSize": 0.1},
            "estimator": GBTClassifier(featuresCol="features", labelCol="label", maxDepth=5, maxIter=50, stepSize=0.1, seed=42)
        }
    )

print("Model configurations:")
for config in model_configs:
    print(config["run_name"], config["model_type"], config["params"])


## 4.4 Evaluation

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion matrix
- Feature importance

In [ ]:
auc_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
precision_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
recall_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

def evaluate_predictions(predictions):
    return {
        "accuracy": accuracy_evaluator.evaluate(predictions),
        "precision": precision_evaluator.evaluate(predictions),
        "recall": recall_evaluator.evaluate(predictions),
        "f1_score": f1_evaluator.evaluate(predictions),
        "roc_auc": auc_evaluator.evaluate(predictions),
    }

def confusion_matrix(predictions):
    return predictions.groupBy("label", "prediction").count().orderBy("label", "prediction")

def get_feature_names(transformed_df, features_col="features"):
    attrs = transformed_df.schema[features_col].metadata.get("ml_attr", {}).get("attrs", {})
    feature_attrs = attrs.get("numeric", []) + attrs.get("binary", [])
    return [item["name"] for item in sorted(feature_attrs, key=lambda x: x["idx"])]

def extract_feature_importance(pipeline_model, predictions):
    fitted_model = pipeline_model.stages[-1]
    feature_names = get_feature_names(predictions)

    if hasattr(fitted_model, "featureImportances"):
        values = fitted_model.featureImportances.toArray().tolist()
    elif hasattr(fitted_model, "coefficients"):
        values = [abs(v) for v in fitted_model.coefficients.toArray().tolist()]
    else:
        return None

    rows = list(zip(feature_names, values))
    return spark.createDataFrame(rows, ["feature", "importance"]).orderBy(F.desc("importance"))


## 4.5 Experiment Tracking

- Log parameters, metrics, and timestamps
- Store runs in MLflow
- Compare experiments

In [ ]:
results = []
best_model = None
best_pipeline_model = None
best_predictions = None
best_auc = -1.0

for config in model_configs:
    run_name = config["run_name"]
    estimator = config["estimator"]
    pipeline = Pipeline(stages=preprocessing_stages + [estimator])

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_param("source_table", source_table)
        mlflow.log_param("model_type", config["model_type"])
        mlflow.log_param("train_row_count", train_df.count())
        mlflow.log_param("test_row_count", test_df.count())
        mlflow.log_param("run_timestamp", datetime.utcnow().isoformat())

        for param_name, param_value in config["params"].items():
            mlflow.log_param(param_name, param_value)

        pipeline_model = pipeline.fit(train_df)
        train_predictions = pipeline_model.transform(train_df)
        test_predictions = pipeline_model.transform(test_df)

        train_metrics = evaluate_predictions(train_predictions)
        test_metrics = evaluate_predictions(test_predictions)

        for metric_name, metric_value in train_metrics.items():
            mlflow.log_metric(f"train_{metric_name}", metric_value)

        for metric_name, metric_value in test_metrics.items():
            mlflow.log_metric(f"test_{metric_name}", metric_value)

        overfit_gap = train_metrics["roc_auc"] - test_metrics["roc_auc"]
        mlflow.log_metric("roc_auc_overfit_gap", overfit_gap)

        mlflow.spark.log_model(pipeline_model, artifact_path="model")

        results.append({
            "run_id": run.info.run_id,
            "run_name": run_name,
            "model_type": config["model_type"],
            "train_accuracy": train_metrics["accuracy"],
            "test_accuracy": test_metrics["accuracy"],
            "test_precision": test_metrics["precision"],
            "test_recall": test_metrics["recall"],
            "test_f1_score": test_metrics["f1_score"],
            "test_roc_auc": test_metrics["roc_auc"],
            "roc_auc_overfit_gap": overfit_gap,
        })

        if test_metrics["roc_auc"] > best_auc:
            best_auc = test_metrics["roc_auc"]
            best_model = config
            best_pipeline_model = pipeline_model
            best_predictions = test_predictions

results_df = spark.createDataFrame(results).orderBy(F.desc("test_roc_auc"))
display(results_df)
print("Best model:", best_model["run_name"], "AUC:", best_auc)


In [ ]:
print("Best model confusion matrix")
display(confusion_matrix(best_predictions))

feature_importance_df = extract_feature_importance(best_pipeline_model, best_predictions)
if feature_importance_df is not None:
    display(feature_importance_df.limit(30))
else:
    print("Feature importance is not available for this model type.")


## Final Notes

Use the MLflow experiment page to compare all runs. A large positive `roc_auc_overfit_gap` means the model performs much better on training data than test data, which is a sign of overfitting. Low train and test metrics together usually suggest underfitting.